# ds003645 — MEG Validation Package

Implements the MEG-EEG NMD validation tests from MEG2.md:
A0, A1, A4, B1, B3, C1, C2, C3, C4, E2, E3, F3, Dashboard.

In [1]:
import sys, json, warnings
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except AttributeError:
    pass  # Jupyter OutStream does not support reconfigure
warnings.filterwarnings('ignore')
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import mne
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr, kendalltau, ttest_ind
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_ROOT  = Path(r'K:/ExternalReceivedDatasets/openneuro/received/ds003645')
OUT_DIR    = Path(r'E:/Science_Datasets/openneuro/processed/ds003645')
RUN_DIR    = sorted(OUT_DIR.glob('neuralmanifolddynamics_*'))[-1]
SAVE_DIR   = OUT_DIR / 'meg_eeg_comparison'
FIG_DIR    = SAVE_DIR / 'figures'
SAVE_DIR.mkdir(exist_ok=True); FIG_DIR.mkdir(exist_ok=True)

SUBS  = ['002', '003', '004', '005', '006']
RUNS  = ['1', '2', '3', '4', '5', '6']
TASK  = 'FacePerception'
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']
CORE_FEATS = ['delta', 'theta', 'alpha', 'beta', 'gamma',
              'hjorth_mobility', 'hjorth_complexity', 'permutation_entropy']
SUBCOORDS = ['m_a', 'm_e', 'm_o', 'd_n', 'd_l', 'd_s', 'e_e', 'e_s', 'e_m']
FAMILIES   = {'m': ['m_a','m_e','m_o'], 'd': ['d_n','d_l','d_s'], 'e': ['e_e','e_s','e_m']}

def h5_path(sub, run):
    stem = f'sub-{sub}_meeg_{TASK}_run-{run}'
    return RUN_DIR / stem / f'{stem}.h5'

print('RUN_DIR:', RUN_DIR)
print('H5 files found:', sum(h5_path(s,r).exists() for s in SUBS for r in RUNS))


RUN_DIR: E:\Science_Datasets\openneuro\processed\ds003645\neuralmanifolddynamics_ds003645_20260626_115843
H5 files found: 30


In [2]:
def _decode(v):
    return v.decode() if isinstance(v, bytes) else v

def _get_fif_mask(f, n_total):
    """Boolean mask selecting FIF (MEG+EEG) rows.
    Uses /row_source/has_meg when present (v2 H5), falls back to positional n//2 split."""
    if 'row_source/has_meg' in f:
        return f['row_source/has_meg'][:].astype(bool)
    mask = np.zeros(n_total, dtype=bool)
    mask[n_total // 2:] = True
    return mask

def load_h5_split(path):
    d = {}
    with h5py.File(path, 'r') as f:
        feat_names = [_decode(n) for n in f['features_raw/names'][:]]
        feat_all   = f['features_raw/values'][:].astype(np.float64)
        robz_all   = f['features_robust_z/values'][:].astype(np.float64)
        n_total    = feat_all.shape[0]

        fif_mask = _get_fif_mask(f, n_total)
        set_mask = ~fif_mask
        d['fif_mask']   = fif_mask
        d['set_mask']   = set_mask
        d['feat_names'] = feat_names
        d['n_fif']      = int(fif_mask.sum())

        d['feat_fif']  = feat_all[fif_mask]
        d['robz_fif']  = robz_all[fif_mask]
        d['feat_set']  = feat_all[set_mask]
        d['robz_set']  = robz_all[set_mask]

        # Transform-aware projection_z (correct for MEG spectral features; log10 applied)
        if 'features_projection_z/values' in f:
            projz_all = f['features_projection_z/values'][:].astype(np.float64)
            d['projz_fif'] = projz_all[fif_mask]
            d['projz_set'] = projz_all[set_mask]
        else:
            d['projz_fif'] = d['robz_fif']
            d['projz_set'] = d['robz_set']

        c9d = f['coords_9d/values'][:].astype(np.float64)
        d['coord_names'] = [_decode(n) for n in f['coords_9d/names'][:]]
        d['c9d_fif'] = c9d[fif_mask]
        d['c9d_set'] = c9d[set_mask]

        m3d = f['mnps_3d'][:].astype(np.float64)
        d['mnps3d_fif'] = m3d[fif_mask]
        d['mnps3d_set'] = m3d[set_mask]

        qc = f['qc/windows/retained_after_qc'][:].astype(bool)
        d['qc_fif'] = qc[fif_mask]
        d['qc_set'] = qc[set_mask]

        ws = f['window_start'][:].astype(np.float64)
        we = f['window_end'][:].astype(np.float64)
        d['ws_fif'] = ws[fif_mask]; d['we_fif'] = we[fif_mask]
        d['ws_set'] = ws[set_mask]; d['we_set'] = we[set_mask]

        if 'row_source/raw_file' in f:
            d['raw_file_fif'] = [_decode(x) for x in f['row_source/raw_file'][:][fif_mask]]
        if 'jacobian/J_hat' in f:
            d['J_hat']     = f['jacobian/J_hat'][:].astype(np.float64)
            d['J_centers'] = f['jacobian/centers'][:]

        d['idx_meg']      = [i for i,n in enumerate(feat_names)
                             if n.startswith('meg_') and '_mag_' not in n and '_grad_' not in n]
        d['idx_meg_mag']  = [i for i,n in enumerate(feat_names) if n.startswith('meg_mag_')]
        d['idx_meg_grad'] = [i for i,n in enumerate(feat_names) if n.startswith('meg_grad_')]
        d['idx_eeg']      = [i for i,n in enumerate(feat_names) if n.startswith('eeg_')]
    return d

records = {}
for sub in SUBS:
    for run in RUNS:
        p = h5_path(sub, run)
        if p.exists():
            records[f'{sub}_{run}'] = load_h5_split(p)
        else:
            print(f'  MISSING sub-{sub} run-{run}')

print(f'Loaded {len(records)} H5 files')


Loaded 30 H5 files


In [3]:
# Load existing CSVs from comparison notebook (if available, else compute fresh)
try:
    all_df     = pd.read_csv(SAVE_DIR / 'all_epochs_features.csv')
    epoch_lbl  = pd.read_csv(SAVE_DIR / 'epoch_condition_labels.csv')
    labeled_df = pd.read_csv(SAVE_DIR / 'labeled_manifold_epochs.csv')
    # Ensure string dtypes
    for df in [all_df, epoch_lbl, labeled_df]:
        if 'sub' in df.columns: df['sub'] = df['sub'].astype(str).str.zfill(3)
        if 'run' in df.columns: df['run'] = df['run'].astype(str)
    print(f'all_df: {all_df.shape}')
    print(f'labeled_df: {labeled_df.shape}')
    print(f'epoch_lbl conditions: {epoch_lbl["condition"].value_counts().to_dict()}')
except FileNotFoundError as e:
    print(f'CSV not found: {e}')
    print('Run ds003645_meg_eeg_comparison.ipynb first to generate CSVs.')
    labeled_df = None; epoch_lbl = None; all_df = None

# Helper: get condition labels for a given sub/run, aligned to FIF window indices
def get_condition_labels(sub, run, n_windows):
    # Return condition array (length n_windows) for FIF-derived epochs
    if epoch_lbl is None: return np.full(n_windows, 'unknown', dtype=object)
    sub_str = str(sub).zfill(3); run_str = str(run)
    grp = epoch_lbl[(epoch_lbl['sub']==sub_str) & (epoch_lbl['run']==run_str)].reset_index(drop=True)
    conds = np.full(n_windows, 'no_stim', dtype=object)
    for i in range(min(n_windows, len(grp))):
        conds[i] = grp.at[i, 'condition']
    return conds


all_df: (7366, 118)
labeled_df: (3683, 67)
epoch_lbl conditions: {'face': 2131, 'scrambled': 788, 'mixed': 590, 'no_stim': 174}


## A0 · Raw file audit

Verify that every FIF contains usable EEG, MAG, and GRAD channels and that
the events TSV exists for every run.

In [4]:
a0_rows = []
for sub in SUBS:
    for run in RUNS:
        fif_p = DATA_ROOT / f'sub-{sub}' / 'meg' / f'sub-{sub}_task-{TASK}_run-{run}_meg.fif'
        ev_p  = DATA_ROOT / f'sub-{sub}' / f'sub-{sub}_task-{TASK}_run-{run}_events.tsv'
        row = {'sub': sub, 'run': run, 'fif_found': fif_p.exists(), 'ev_found': ev_p.exists()}
        if fif_p.exists():
            try:
                raw = mne.io.read_raw_fif(str(fif_p), preload=False, verbose=False)
                row.update({
                    'sfreq':        raw.info['sfreq'],
                    'duration_sec': round(raw.n_times / raw.info['sfreq'], 1),
                    'n_eeg':        len(mne.pick_types(raw.info, eeg=True, meg=False)),
                    'n_mag':        len(mne.pick_types(raw.info, meg='mag')),
                    'n_grad':       len(mne.pick_types(raw.info, meg='grad')),
                    'n_eog':        len(mne.pick_types(raw.info, eog=True)),
                    'n_ecg':        len(mne.pick_types(raw.info, ecg=True)),
                })
                row['pass_ch'] = row['n_eeg'] > 0 and row['n_mag'] > 0 and row['n_grad'] > 0
            except Exception as e:
                row['error'] = str(e)[:80]
        if ev_p.exists():
            ev = pd.read_csv(ev_p, sep='\t')
            row['n_events'] = len(ev)
            if 'onset' in ev.columns:
                row['ev_onset_min'] = round(float(ev['onset'].min()), 2)
                row['ev_onset_max'] = round(float(ev['onset'].max()), 2)
        a0_rows.append(row)

a0_df = pd.DataFrame(a0_rows)
a0_df.to_csv(SAVE_DIR / 'a0_raw_audit.csv', index=False)
disp_cols = [c for c in ['sub','run','n_eeg','n_mag','n_grad','n_eog','duration_sec','n_events','pass_ch'] if c in a0_df.columns]
print(a0_df[disp_cols].to_string(index=False))
n_pass = a0_df.get('pass_ch', pd.Series([True]*len(a0_df))).sum()
print(f'\nA0 PASS: {n_pass}/{len(a0_df)} runs have EEG+MAG+GRAD')


sub run  n_eeg  n_mag  n_grad  n_eog  duration_sec  n_events  pass_ch
002   1     74    102     204      0         491.0       552     True
002   2     74    102     204      0         497.0       516     True
002   3     74    102     204      0         505.0       533     True
002   4     74    102     204      0         493.0       532     True
002   5     74    102     204      0         505.0       546     True
002   6     74    102     204      0         500.0       575     True
003   1     74    102     204      0         494.0       589     True
003   2     74    102     204      0         493.0       589     True
003   3     74    102     204      0         492.0       583     True
003   4     74    102     204      0         503.0       584     True
003   5     74    102     204      0         494.0       579     True
003   6     74    102     204      0         490.0       588     True
004   1     74    102     204      0         506.0       589     True
004   2     74    10

## A1 · Timebase and event-grid audit

Verify window counts, condition label distribution, and that event onsets
fall within recording bounds.

In [5]:
a1_rows = []
if labeled_df is not None:
    for (sub, run), grp in labeled_df.groupby(['sub', 'run']):
        sub_s = str(sub).zfill(3); run_s = str(run)
        # duration from A0
        a0_r = a0_df[(a0_df['sub']==sub_s) & (a0_df['run'].astype(str)==run_s)] if 'a0_df' in dir() else pd.DataFrame()
        dur  = float(a0_r['duration_sec'].iloc[0]) if (not a0_r.empty and 'duration_sec' in a0_r.columns) else np.nan
        cond_counts = grp['condition'].value_counts().to_dict() if 'condition' in grp.columns else {}
        ws_col = 'window_start' if 'window_start' in grp.columns else None
        a1_rows.append({
            'sub': sub_s, 'run': run_s,
            'n_windows': len(grp),
            'n_face':     cond_counts.get('face', 0),
            'n_scrambled':cond_counts.get('scrambled', 0),
            'n_mixed':    cond_counts.get('mixed', 0),
            'n_no_stim':  cond_counts.get('no_stim', 0),
            'ws_min':     round(float(grp[ws_col].min()), 2) if ws_col else np.nan,
            'ws_max':     round(float(grp[ws_col].max()), 2) if ws_col else np.nan,
            'duration_sec': round(dur, 1),
            'ws_within_recording': (float(grp[ws_col].max()) < dur) if (ws_col and np.isfinite(dur)) else True,
        })

a1_df = pd.DataFrame(a1_rows)
a1_df.to_csv(SAVE_DIR / 'a1_timebase_audit.csv', index=False)
print(a1_df.to_string(index=False))
if 'ws_within_recording' in a1_df.columns:
    print(f'\nA1 PASS (all windows within recording): {a1_df["ws_within_recording"].all()}')
else:
    print('A1: duration data unavailable (run A0 first)')


sub run  n_windows  n_face  n_scrambled  n_mixed  n_no_stim  ws_min  ws_max  duration_sec  ws_within_recording
002   1        121      69           29       18          5     NaN     NaN         491.0                 True
002   2        123      68           27       22          6     NaN     NaN         497.0                 True
002   3        125      75           26       19          5     NaN     NaN         505.0                 True
002   4        122      71           27       19          5     NaN     NaN         493.0                 True
002   5        125      74           26       18          7     NaN     NaN         505.0                 True
002   6        124      71           30       17          6     NaN     NaN         500.0                 True
003   1        122      67           30       19          6     NaN     NaN         494.0                 True
003   2        122      67           28       21          6     NaN     NaN         493.0                 True
0

## A4 · NMD HDF5 contract test

Verify all required paths exist in every H5 output file.

In [6]:
REQUIRED = [
    'mnps_3d', 'coords_9d/values', 'coords_9d/names',
    'jacobian/J_hat', 'features_raw/values', 'features_robust_z/values',
    'window_start', 'window_end', 'qc/windows/retained_after_qc',
    'features_raw/names',
]
a4_rows = []
for sub in SUBS:
    for run in RUNS:
        p = h5_path(sub, run)
        row = {'sub': sub, 'run': run, 'h5_found': p.exists(), 'pass_all': False}
        if p.exists():
            with h5py.File(p, 'r') as f:
                checks = {rp: (rp in f) for rp in REQUIRED}
                row.update({f'has_{k.replace("/","_")}': v for k, v in checks.items()})
                row['pass_all'] = all(checks.values())
                if 'jacobian/J_hat' in f:
                    J = f['jacobian/J_hat'][:].astype(np.float64)
                    row['n_jacobians'] = J.shape[0]
                    row['jacobian_finite_frac'] = round(float(np.isfinite(J).all(axis=(1,2)).mean()), 4)
        a4_rows.append(row)

a4_df = pd.DataFrame(a4_rows)
a4_df.to_csv(SAVE_DIR / 'a4_h5_contract.csv', index=False)
n_pass = a4_df['pass_all'].sum()
print(f'A4 PASS: {n_pass}/{len(a4_df)} H5 files have all required paths')
if 'jacobian_finite_frac' in a4_df.columns:
    print(f'Mean Jacobian finite fraction: {a4_df["jacobian_finite_frac"].mean():.4f}')
print(a4_df[['sub','run','pass_all','n_jacobians']].to_string(index=False))


A4 PASS: 30/30 H5 files have all required paths
Mean Jacobian finite fraction: 1.0000
sub run  pass_all  n_jacobians
002   1      True          121
002   2      True          123
002   3      True          125
002   4      True          122
002   5      True          125
002   6      True          124
003   1      True          122
003   2      True          122
003   3      True          122
003   4      True          124
003   5      True          122
003   6      True          121
004   1      True          125
004   2      True          122
004   3      True          125
004   4      True          123
004   5      True          122
004   6      True          124
005   1      True          123
005   2      True          121
005   3      True          124
005   4      True          122
005   5      True          122
005   6      True          123
006   1      True          121
006   2      True          125
006   3      True          123
006   4      True          120
006   5      Tr

## B1 · MAG vs GRAD feature agreement

Compare magnetometer and gradiometer features separately. Pass if the same
face–scrambled direction holds in ≥60% of core features for both sensor families.

In [7]:
b1_rows = []
for key, d in records.items():
    sub, run = key.split('_')
    fnames   = d['feat_names']
    feat     = d['feat_fif']
    qc       = d['qc_fif']
    ws       = d['ws_fif']
    conds    = get_condition_labels(sub, run, len(feat))

    mag_names  = [n for n in fnames if n.startswith('meg_mag_')]
    grad_names = [n.replace('meg_mag_', 'meg_grad_') for n in mag_names
                  if n.replace('meg_mag_', 'meg_grad_') in fnames]

    face_m = conds == 'face'; scr_m = conds == 'scrambled'
    for mn, gn in zip(mag_names, grad_names):
        mi = fnames.index(mn); gi = fnames.index(gn)
        mv = np.log10(np.abs(feat[:, mi]) + 1e-30)
        gv = np.log10(np.abs(feat[:, gi]) + 1e-30)
        valid = np.isfinite(mv) & np.isfinite(gv)
        if valid.sum() < 10: continue
        r_pg, _ = pearsonr(mv[valid], gv[valid])
        # Face-scrambled sign agreement
        if face_m.sum() >= 3 and scr_m.sum() >= 3:
            d_mag  = np.nanmean(mv[face_m]) - np.nanmean(mv[scr_m])
            d_grad = np.nanmean(gv[face_m]) - np.nanmean(gv[scr_m])
            sign_agree = int(np.sign(d_mag) == np.sign(d_grad))
        else:
            sign_agree = np.nan
        feat_name = mn.replace('meg_mag_', '')
        b1_rows.append({'sub': sub, 'run': run, 'feature': feat_name,
                        'r_mag_grad': round(r_pg, 4), 'sign_agree': sign_agree})

b1_df = pd.DataFrame(b1_rows)
b1_df.to_csv(SAVE_DIR / 'b1_mag_grad_agreement.csv', index=False)
summary = b1_df.groupby('feature')[['r_mag_grad', 'sign_agree']].mean().round(3)
print(summary.to_string())
mean_r    = b1_df['r_mag_grad'].mean()
sign_rate = b1_df['sign_agree'].mean()
print(f'\nMean MAG-GRAD corr: {mean_r:.3f}')
print(f'Mean sign agreement: {sign_rate:.3f}')
b1_pass = sign_rate >= 0.60
print(f'B1 PASS (sign_agree >= 0.60): {b1_pass}')


                      r_mag_grad  sign_agree
feature                                     
alpha                      0.295       0.367
alpha_theta                0.479       0.600
beta                       0.582       0.733
beta_alpha                 0.374       0.500
delta                      0.705       0.767
gamma                      0.224       0.567
highfreq_power_30_45       0.224       0.567
hjorth_complexity          0.678       0.867
hjorth_mobility            0.729       0.867
permutation_entropy        0.137       0.567
sample_entropy             0.424       0.867
theta                      0.594       0.767

Mean MAG-GRAD corr: 0.454
Mean sign agreement: 0.669
B1 PASS (sign_agree >= 0.60): True


## B3 · MEG spectral sanity

Check that MEG band powers are not constant, not dominated by a single extreme
feature, and that the first PC does not capture >95% of variance (broadband dominance).

In [8]:
b3_rows = []
for key, d in records.items():
    sub, run = key.split('_')
    fnames = d['feat_names']
    feat   = d['feat_fif']

    band_idxs = [fnames.index(f'meg_{b}') for b in BANDS if f'meg_{b}' in fnames]
    if not band_idxs: continue
    band_mat = feat[:, band_idxs]
    valid_rows = np.isfinite(band_mat).all(axis=1)
    bm = np.log10(np.abs(band_mat[valid_rows]) + 1e-30)
    if bm.shape[0] < 5: continue

    # Per-band stats
    for j, b in enumerate(BANDS):
        if j >= bm.shape[1]: continue
        col = bm[:, j]
        b3_rows.append({
            'sub': sub, 'run': run, 'band': b,
            'mean': round(float(np.mean(col)), 4),
            'std':  round(float(np.std(col)), 4),
            'min':  round(float(np.min(col)), 4),
            'max':  round(float(np.max(col)), 4),
            'is_constant': float(np.std(col)) < 1e-6,
        })

    # PCA on band matrix
    pca = PCA()
    pca.fit(bm)
    pc1_var = float(pca.explained_variance_ratio_[0])

    # Gamma outlier fraction (>3 SD)
    if 'meg_gamma' in fnames:
        gi = fnames.index('meg_gamma')
        gv = np.log10(np.abs(feat[:, gi][valid_rows]) + 1e-30)
        gm, gs = gv.mean(), gv.std()
        outlier_frac = float(np.mean(np.abs(gv - gm) > 3 * gs)) if gs > 0 else 0.0
    else:
        outlier_frac = np.nan

    for r in b3_rows[-len(BANDS):]:
        r['pc1_var'] = round(pc1_var, 4)
        r['gamma_outlier_frac'] = round(outlier_frac, 4) if np.isfinite(outlier_frac) else np.nan

b3_df = pd.DataFrame(b3_rows)
b3_df.to_csv(SAVE_DIR / 'b3_spectral_sanity.csv', index=False)
print(b3_df.groupby('band')[['mean','std','is_constant']].mean().round(3).to_string())
if 'pc1_var' in b3_df.columns:
    print(f'\nMean first-PC variance explained: {b3_df["pc1_var"].mean():.3f}')
print(f'Mean gamma outlier fraction: {b3_df["gamma_outlier_frac"].mean():.4f}' if 'gamma_outlier_frac' in b3_df.columns else '')
b3_pass = not b3_df['is_constant'].any()
print(f'B3 PASS (no constant features): {b3_pass}')


         mean    std  is_constant
band                             
alpha -25.011  0.087          0.0
beta  -24.701  0.086          0.0
delta -24.469  0.160          0.0
gamma -25.710  0.060          0.0
theta -24.854  0.117          0.0

Mean first-PC variance explained: 0.532
Mean gamma outlier fraction: 0.0087
B3 PASS (no constant features): True


## C1 · Event-response vector agreement (central test)

For each subject/run, compute the face-scrambled difference vector in
robust-z feature space for MEG and EEG separately, then measure their
cosine similarity. Compare to event-label shuffle null (N=1000).

In [9]:
def delta_vec(robz_mat, conds, feats_idx, condition='face', reference='scrambled'):
    # Compute mean_condition - mean_reference in robust-z feature space
    f_rows = conds == condition; s_rows = conds == reference
    if f_rows.sum() < 3 or s_rows.sum() < 3: return None
    fv = np.nanmean(robz_mat[f_rows][:, feats_idx], axis=0)
    sv = np.nanmean(robz_mat[s_rows][:, feats_idx], axis=0)
    delta = fv - sv
    if not np.any(np.isfinite(delta)): return None
    return delta

c1_rows = []
rng = np.random.default_rng(42)
N_SHUFFLE = 500

for key, d in records.items():
    sub, run = key.split('_')
    fnames   = d['feat_names']
    robz_fif = d['robz_fif']
    robz_set = d['robz_set']
    conds    = get_condition_labels(sub, run, len(robz_fif))

    # Feature index lists for core features
    meg_idx = [fnames.index(f'meg_{f}') for f in CORE_FEATS if f'meg_{f}' in fnames]
    eeg_idx = [fnames.index(f'eeg_{f}') for f in CORE_FEATS if f'eeg_{f}' in fnames]
    if not meg_idx or not eeg_idx: continue

    d_meg = delta_vec(robz_fif, conds, meg_idx)
    # EEG from .set rows (same condition labels, same window order)
    d_eeg = delta_vec(robz_set, conds, eeg_idx)
    if d_meg is None or d_eeg is None: continue

    # Cosine similarity
    def cosine(a, b):
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        if na < 1e-12 or nb < 1e-12: return np.nan
        return float(np.dot(a, b) / (na * nb))

    obs_cos = cosine(d_meg, d_eeg)

    # Null: shuffle condition labels within run
    null_cos = []
    for _ in range(N_SHUFFLE):
        shuf = conds[rng.permutation(len(conds))]
        dm_s = delta_vec(robz_fif, shuf, meg_idx)
        de_s = delta_vec(robz_set, shuf, eeg_idx)
        if dm_s is not None and de_s is not None:
            null_cos.append(cosine(dm_s, de_s))
    null_arr = np.array(null_cos)
    p_val    = float(np.mean(null_arr >= obs_cos)) if len(null_arr) else np.nan

    c1_rows.append({
        'sub': sub, 'run': run,
        'obs_cosine': round(obs_cos, 4),
        'null_mean':  round(float(np.mean(null_arr)), 4) if len(null_arr) else np.nan,
        'null_std':   round(float(np.std(null_arr)), 4) if len(null_arr) else np.nan,
        'p_vs_null':  round(p_val, 4),
        'n_face':     int((conds == 'face').sum()),
        'n_scr':      int((conds == 'scrambled').sum()),
    })

c1_df = pd.DataFrame(c1_rows)
c1_df.to_csv(SAVE_DIR / 'c1_event_response_agreement.csv', index=False)
print(c1_df.to_string(index=False))
mean_cos = c1_df['obs_cosine'].mean()
mean_null = c1_df['null_mean'].mean()
print(f'\nMean observed cosine: {mean_cos:.4f}  |  Mean null: {mean_null:.4f}')
c1_pass = mean_cos > mean_null
print(f'C1 PASS (mean obs > mean null): {c1_pass}')


sub run  obs_cosine  null_mean  null_std  p_vs_null  n_face  n_scr
002   1      0.6070    -0.0166    0.4538      0.094      69     29
002   2      0.1550     0.0199    0.3979      0.398      68     27
002   3      0.1898    -0.0427    0.4034      0.328      75     26
002   4      0.7630     0.0585    0.4230      0.024      71     27
002   5     -0.6831     0.0434    0.4268      0.974      74     26
002   6      0.4308     0.0825    0.2928      0.128      71     30
003   1      0.7346    -0.0282    0.4142      0.036      67     30
003   2      0.0055     0.0895    0.3882      0.586      67     28
003   3     -0.5330     0.0134    0.4037      0.906      71     24
003   4     -0.4266    -0.0458    0.3232      0.886      69     25
003   5      0.4978     0.0398    0.2949      0.074      69     24
003   6     -0.2326    -0.0184    0.3460      0.692      71     27
004   1     -0.0297     0.0205    0.3375      0.572      76     26
004   2     -0.2296     0.0091    0.3376      0.740      75   

## C2 · Subcoordinate family sign agreement

For each subject/run, compute face-scrambled delta per 9D subcoordinate
using MEG-derived 9D (FIF rows) and EEG-computed 9D (using EEG features
with the same 9D mapping formula). Compare family-level signs.

In [10]:
# EEG 9D mapping: same structure as MEG but using eeg_* features
EEG_9D_MAP = {
    'm_a': {'eeg_delta': -0.5, 'eeg_theta': -0.5},
    'm_e': {'eeg_alpha': -1.0},
    'm_o': {'eeg_beta_alpha': 1.0},
    'd_n': {'eeg_gamma': 1.0},
    'd_l': {'eeg_hjorth_mobility': 1.0},
    'd_s': {'eeg_alpha_theta': 1.0},
    'e_e': {'eeg_permutation_entropy': 1.0},
    'e_s': {'eeg_hjorth_complexity': 1.0},
    'e_m': {},  # no ECG proxy; leave empty
}

def compute_9d_from_robz(robz_mat, fnames, mapping):
    # Compute 9D subcoords from robust-z features using a mapping dict
    result = {}
    for sc, weights in mapping.items():
        col = np.zeros(len(robz_mat))
        total_w = 0.0
        for feat, w in weights.items():
            if feat in fnames:
                idx = fnames.index(feat)
                v = robz_mat[:, idx]
                col += w * np.where(np.isfinite(v), v, 0.0)
                total_w += abs(w)
        result[sc] = col / total_w if total_w > 0 else np.full(len(robz_mat), np.nan)
    return result

c2_rows = []
for key, d in records.items():
    sub, run = key.split('_')
    fnames   = d['feat_names']
    conds    = get_condition_labels(sub, run, d['n_fif'])
    face_m   = conds == 'face'; scr_m = conds == 'scrambled'
    if face_m.sum() < 3 or scr_m.sum() < 3: continue

    # MEG 9D from H5 (FIF rows)
    meg_9d = {name: d['c9d_fif'][:, i] for i, name in enumerate(d['coord_names'])}

    # EEG 9D computed from .set robust-z
    eeg_9d_raw = compute_9d_from_robz(d['robz_set'], fnames, EEG_9D_MAP)

    for sc in SUBCOORDS:
        meg_v = meg_9d.get(sc)
        eeg_v = eeg_9d_raw.get(sc)
        if meg_v is None or eeg_v is None: continue
        if not np.any(np.isfinite(meg_v)) or not np.any(np.isfinite(eeg_v)): continue

        d_meg = np.nanmean(meg_v[face_m]) - np.nanmean(meg_v[scr_m])
        d_eeg = np.nanmean(eeg_v[face_m]) - np.nanmean(eeg_v[scr_m])
        sign_agree = int(np.sign(d_meg) == np.sign(d_eeg)) if (np.isfinite(d_meg) and np.isfinite(d_eeg)) else np.nan
        family = [k for k, v in FAMILIES.items() if sc in v][0]
        c2_rows.append({'sub': sub, 'run': run, 'subcoord': sc, 'family': family,
                        'delta_meg': round(d_meg, 5), 'delta_eeg': round(d_eeg, 5),
                        'sign_agree': sign_agree})

c2_df = pd.DataFrame(c2_rows)
c2_df.to_csv(SAVE_DIR / 'c2_family_sign_agreement.csv', index=False)
summary_c2 = c2_df.groupby(['family', 'subcoord'])[['delta_meg','delta_eeg','sign_agree']].mean().round(4)
print(summary_c2.to_string())
mean_sign = c2_df['sign_agree'].mean()
print(f'\nOverall sign agreement: {mean_sign:.3f}')
print(f'C2 PASS (mean sign agree > 0.5): {mean_sign > 0.5}')


                 delta_meg  delta_eeg  sign_agree
family subcoord                                  
d      d_l         -0.3926    -0.1162      0.5000
       d_n          0.0000     0.0719      0.0000
       d_s         -0.0015     0.0816      0.5333
e      e_e          0.0009    -0.0500      0.6333
       e_s          0.3823     0.0331      0.4000
m      m_a          0.0000     0.0132      0.0000
       m_e          0.0000    -0.0176      0.0000
       m_o         -0.0112     0.0342      0.4667

Overall sign agreement: 0.317
C2 PASS (mean sign agree > 0.5): False


## C3 · Temporal co-variation with window-shift tolerance

Compute EEG-MEG feature correlations across window-index shifts (±1, ±2 steps = ±4s, ±8s).
Find the best lag per feature; pass if gamma/e_m beat temporal null.

In [11]:
c3_rows = []
SHIFTS = [-2, -1, 0, 1, 2]  # window-index shifts; 1 step = 4 s

for key, d in records.items():
    sub, run = key.split('_')
    fnames   = d['feat_names']
    feat_fif = d['robz_fif']
    feat_set = d['robz_set']

    for feat_name in CORE_FEATS:
        meg_col = f'meg_{feat_name}'
        eeg_col = f'eeg_{feat_name}'
        if meg_col not in fnames or eeg_col not in fnames: continue
        mi = fnames.index(meg_col); ei = fnames.index(eeg_col)
        mv = feat_fif[:, mi]; ev = feat_set[:, ei]

        r_by_shift = {}
        for shift in SHIFTS:
            if shift == 0:
                a, b = mv, ev
            elif shift > 0:
                a, b = mv[shift:], ev[:-shift]
            else:
                a, b = mv[:shift], ev[-shift:]
            valid = np.isfinite(a) & np.isfinite(b)
            if valid.sum() < 10:
                r_by_shift[shift] = np.nan; continue
            r_val, _ = pearsonr(a[valid], b[valid])
            r_by_shift[shift] = r_val

        r_vals = [r_by_shift[s] for s in SHIFTS if np.isfinite(r_by_shift.get(s, np.nan))]
        if not r_vals: continue
        max_r   = max(r_vals)
        best_sh = SHIFTS[np.argmax([r_by_shift.get(s, -np.inf) for s in SHIFTS])]

        c3_rows.append({'sub': sub, 'run': run, 'feature': feat_name,
                        'r_shift_0':  round(r_by_shift.get(0, np.nan), 4),
                        'max_r_any':  round(max_r, 4),
                        'best_shift': best_sh,
                        **{f'r_shift_{s}': round(r_by_shift.get(s, np.nan), 4) for s in SHIFTS}})

c3_df = pd.DataFrame(c3_rows)
c3_df.to_csv(SAVE_DIR / 'c3_lagged_correlations.csv', index=False)
print(c3_df.groupby('feature')[['r_shift_0', 'max_r_any', 'best_shift']].mean().round(3).to_string())


                     r_shift_0  max_r_any  best_shift
feature                                              
alpha                    0.212      0.239       0.200
beta                     0.138      0.168      -0.300
delta                    0.070      0.113       0.333
gamma                    0.186      0.211      -0.067
hjorth_complexity        0.018      0.090       0.167
hjorth_mobility          0.030      0.097       0.000
permutation_entropy      0.012      0.102      -0.133
theta                    0.008      0.097      -0.200


## C4 · Rank-order condition agreement

For each run, rank {face, scrambled, mixed, no_stim} by mean feature value
for MEG and EEG separately. Compute Spearman tau for MEG vs EEG rank ordering.

In [12]:
c4_rows = []
CONDITIONS = ['face', 'scrambled', 'mixed', 'no_stim']

for key, d in records.items():
    sub, run = key.split('_')
    fnames   = d['feat_names']
    robz_fif = d['robz_fif']
    robz_set = d['robz_set']
    conds    = get_condition_labels(sub, run, d['n_fif'])

    for feat_name in CORE_FEATS:
        mc = f'meg_{feat_name}'; ec = f'eeg_{feat_name}'
        if mc not in fnames or ec not in fnames: continue
        mi = fnames.index(mc); ei = fnames.index(ec)

        meg_means = [np.nanmean(robz_fif[conds==c, mi]) if (conds==c).sum()>=2 else np.nan for c in CONDITIONS]
        eeg_means = [np.nanmean(robz_set[conds==c, ei]) if (conds==c).sum()>=2 else np.nan for c in CONDITIONS]

        valid = [i for i in range(4) if np.isfinite(meg_means[i]) and np.isfinite(eeg_means[i])]
        if len(valid) < 3: continue

        m_v = [meg_means[i] for i in valid]
        e_v = [eeg_means[i] for i in valid]
        m_rank = np.argsort(np.argsort(m_v)).tolist()
        e_rank = np.argsort(np.argsort(e_v)).tolist()
        if len(m_rank) >= 2:
            tau, p_tau = kendalltau(m_rank, e_rank)
        else:
            tau, p_tau = np.nan, np.nan

        c4_rows.append({'sub': sub, 'run': run, 'feature': feat_name,
                        'kendall_tau': round(tau, 4), 'p_tau': round(p_tau, 4),
                        'n_conditions': len(valid)})

c4_df = pd.DataFrame(c4_rows)
c4_df.to_csv(SAVE_DIR / 'c4_rank_agreement.csv', index=False)
print(c4_df.groupby('feature')[['kendall_tau', 'p_tau']].mean().round(3).to_string())
mean_tau = c4_df['kendall_tau'].mean()
print(f'\nMean Kendall tau: {mean_tau:.3f}')
print(f'C4 PASS (mean tau > 0): {mean_tau > 0}')


                     kendall_tau  p_tau
feature                                
alpha                      0.244  0.528
beta                       0.156  0.461
delta                      0.011  0.581
gamma                      0.256  0.564
hjorth_complexity         -0.044  0.494
hjorth_mobility           -0.056  0.525
permutation_entropy       -0.089  0.694
theta                     -0.044  0.594

Mean Kendall tau: 0.054
C4 PASS (mean tau > 0): True


## E2 · Temporal circular-shift null

Circularly shift MEG window indices within each run. True synchrony
(shift=0) should exceed the distribution across shifts.

In [13]:
e2_rows = []
rng2 = np.random.default_rng(99)

for key, d in records.items():
    sub, run = key.split('_')
    fnames   = d['feat_names']
    robz_fif = d['robz_fif']
    robz_set = d['robz_set']
    n        = len(robz_fif)
    if n < 10: continue

    for feat_name in BANDS:
        mc = f'meg_{feat_name}'; ec = f'eeg_{feat_name}'
        if mc not in fnames or ec not in fnames: continue
        mi = fnames.index(mc); ei = fnames.index(ec)
        mv = robz_fif[:, mi]; ev = robz_set[:, ei]
        valid = np.isfinite(mv) & np.isfinite(ev)
        if valid.sum() < 10: continue

        # Observed correlation at shift=0
        r_obs, _ = pearsonr(mv[valid], ev[valid])

        # Null: circular shifts of MEG
        shifts = np.arange(1, n)
        null_r = []
        for sh in rng2.choice(shifts, size=min(200, len(shifts)), replace=False):
            mv_sh = np.roll(mv, int(sh))
            v2 = np.isfinite(mv_sh) & np.isfinite(ev)
            if v2.sum() < 10: continue
            r_sh, _ = pearsonr(mv_sh[v2], ev[v2])
            null_r.append(r_sh)

        null_arr = np.array(null_r)
        p_null   = float(np.mean(null_arr >= r_obs)) if len(null_arr) else np.nan
        e2_rows.append({'sub': sub, 'run': run, 'band': feat_name,
                        'r_obs':    round(r_obs, 4),
                        'null_mean':round(float(np.mean(null_arr)), 4) if len(null_arr) else np.nan,
                        'null_std': round(float(np.std(null_arr)), 4) if len(null_arr) else np.nan,
                        'p_vs_null':round(p_null, 4)})

e2_df = pd.DataFrame(e2_rows)
e2_df.to_csv(SAVE_DIR / 'e2_temporal_shift_null.csv', index=False)
print(e2_df.groupby('band')[['r_obs','null_mean','p_vs_null']].mean().round(3).to_string())
sig_bands = e2_df[e2_df['p_vs_null'] < 0.05]['band'].unique().tolist()
print(f'\nE2: Bands beating temporal null (p<0.05): {sig_bands}')
print(f'E2 PASS (at least one band significant): {len(sig_bands) > 0}')


       r_obs  null_mean  p_vs_null
band                              
alpha  0.212     -0.002      0.191
beta   0.138     -0.001      0.263
delta  0.070     -0.001      0.309
gamma  0.186     -0.002      0.250
theta  0.008     -0.000      0.491

E2: Bands beating temporal null (p<0.05): ['alpha', 'gamma', 'beta', 'delta', 'theta']
E2 PASS (at least one band significant): True


## E3 · Wrong-run pairing null

Pair EEG from run N with MEG from run N+1 (same subject, wrong run).
True-pair agreement should exceed wrong-pair agreement.

In [14]:
e3_rows = []
for sub in SUBS:
    for run_i, run_a in enumerate(RUNS[:-1]):
        run_b = RUNS[run_i + 1]
        key_a = f'{sub}_{run_a}'; key_b = f'{sub}_{run_b}'
        if key_a not in records or key_b not in records: continue

        da = records[key_a]; db = records[key_b]
        fnames = da['feat_names']

        for feat_name in BANDS:
            mc = f'meg_{feat_name}'; ec = f'eeg_{feat_name}'
            if mc not in fnames or ec not in fnames: continue
            mi = fnames.index(mc); ei = fnames.index(ec)

            # True pair: MEG run_a, EEG run_a
            mv_true = da['robz_fif'][:, mi]; ev_true = da['robz_set'][:, ei]
            n_min = min(len(mv_true), len(ev_true))
            v_t = np.isfinite(mv_true[:n_min]) & np.isfinite(ev_true[:n_min])
            if v_t.sum() < 5: continue
            r_true, _ = pearsonr(mv_true[:n_min][v_t], ev_true[:n_min][v_t])

            # Wrong pair: MEG run_b, EEG run_a
            mv_wrong = db['robz_fif'][:, mi]
            n_min2 = min(len(mv_wrong), len(ev_true))
            v_w = np.isfinite(mv_wrong[:n_min2]) & np.isfinite(ev_true[:n_min2])
            if v_w.sum() < 5: continue
            r_wrong, _ = pearsonr(mv_wrong[:n_min2][v_w], ev_true[:n_min2][v_w])

            e3_rows.append({'sub': sub, 'run_eeg': run_a, 'run_meg_wrong': run_b,
                            'band': feat_name,
                            'r_true_pair': round(r_true, 4),
                            'r_wrong_pair': round(r_wrong, 4),
                            'true_gt_wrong': r_true > r_wrong})

e3_df = pd.DataFrame(e3_rows)
e3_df.to_csv(SAVE_DIR / 'e3_wrongrun_null.csv', index=False)
print(e3_df.groupby('band')[['r_true_pair','r_wrong_pair','true_gt_wrong']].mean().round(3).to_string())
true_gt_frac = e3_df['true_gt_wrong'].mean()
print(f'\nFraction where true pair > wrong pair: {true_gt_frac:.3f}')
print(f'E3 PASS (true_gt_wrong > 0.5): {true_gt_frac > 0.5}')


       r_true_pair  r_wrong_pair  true_gt_wrong
band                                           
alpha        0.226         0.031           0.80
beta         0.146         0.043           0.68
delta        0.084         0.023           0.64
gamma        0.183         0.030           0.80
theta        0.004         0.047           0.48

Fraction where true pair > wrong pair: 0.680
E3 PASS (true_gt_wrong > 0.5): True


## F3 · Gamma proxy audit

Audit `e_m` (embodied arousal proxy → `meg_highfreq_power_30_45` fallback)
and `meg_gamma`. Check face-scrambled direction; verify effect is not driven
by artifact (high-frequency outliers).

In [15]:
f3_rows = []
GAMMA_FEATS = ['meg_gamma', 'meg_highfreq_power_30_45', 'meg_permutation_entropy']

for key, d in records.items():
    sub, run = key.split('_')
    fnames  = d['feat_names']
    feat    = d['feat_fif']
    c9d     = d['c9d_fif']
    qc      = d['qc_fif']
    cnames  = d['coord_names']
    conds   = get_condition_labels(sub, run, d['n_fif'])

    face_m = conds == 'face'; scr_m = conds == 'scrambled'
    if face_m.sum() < 3 or scr_m.sum() < 3: continue

    # e_m from 9D coords
    if 'e_m' in cnames:
        em_idx = cnames.index('e_m')
        em_v = c9d[:, em_idx]
        # Face-scrambled delta for e_m
        d_em = float(np.nanmean(em_v[face_m]) - np.nanmean(em_v[scr_m]))
        # After QC filtering
        qc_mask = qc & (face_m | scr_m)
        if qc_mask.sum() >= 6:
            d_em_qc = float(np.nanmean(em_v[qc & face_m]) - np.nanmean(em_v[qc & scr_m]))
        else:
            d_em_qc = np.nan
    else:
        d_em = np.nan; d_em_qc = np.nan

    row_base = {'sub': sub, 'run': run, 'e_m_delta': round(d_em, 5),
                'e_m_delta_qc': round(d_em_qc, 5) if np.isfinite(d_em_qc) else np.nan,
                'e_m_face_positive': d_em > 0 if np.isfinite(d_em) else np.nan}

    for gf in GAMMA_FEATS:
        if gf not in fnames: continue
        gi = fnames.index(gf)
        gv = np.log10(np.abs(feat[:, gi]) + 1e-30)
        # Gamma outlier mask (>3 SD)
        gm, gs = np.nanmean(gv), np.nanstd(gv)
        hi_art = np.abs(gv - gm) > 3 * gs
        clean = ~hi_art
        d_all   = float(np.nanmean(gv[face_m]) - np.nanmean(gv[scr_m]))
        d_clean = float(np.nanmean(gv[face_m & clean]) - np.nanmean(gv[scr_m & clean])) \
                  if (face_m & clean).sum() >= 3 and (scr_m & clean).sum() >= 3 else np.nan
        row_base[f'{gf}_delta_all']   = round(d_all, 5)
        row_base[f'{gf}_delta_clean'] = round(d_clean, 5) if np.isfinite(d_clean) else np.nan
        row_base[f'{gf}_artifact_frac'] = round(float(hi_art.mean()), 4)
        row_base[f'{gf}_sign_preserved'] = (np.sign(d_all) == np.sign(d_clean)) if np.isfinite(d_clean) else np.nan

    f3_rows.append(row_base)

f3_df = pd.DataFrame(f3_rows)
f3_df.to_csv(SAVE_DIR / 'f3_gamma_proxy_audit.csv', index=False)
disp_cols = [c for c in f3_df.columns if c in ['sub','run','e_m_delta','e_m_delta_qc',
             'e_m_face_positive','meg_gamma_delta_all','meg_gamma_delta_clean',
             'meg_gamma_artifact_frac','meg_gamma_sign_preserved']]
print(f3_df[disp_cols].to_string(index=False))
if 'e_m_face_positive' in f3_df.columns:
    print(f'\nFraction runs with e_m face > scrambled: {f3_df["e_m_face_positive"].mean():.3f}')


sub run  e_m_delta  e_m_delta_qc  e_m_face_positive  meg_gamma_delta_all  meg_gamma_delta_clean  meg_gamma_artifact_frac  meg_gamma_sign_preserved
002   1   -0.22555      -0.22555              False              0.01667                0.01667                   0.0000                      True
002   2   -0.49325      -0.49325              False             -0.00062               -0.00346                   0.0163                      True
002   3   -0.19800      -0.19800              False              0.00951                0.00951                   0.0000                      True
002   4   -0.51062      -0.51062              False              0.02244                0.02244                   0.0000                      True
002   5   -0.51905      -0.51905              False             -0.00059               -0.00059                   0.0000                      True
002   6   -0.03867      -0.03867              False              0.01861                0.01861                   0.01

## Dashboard · MEG readiness score

Aggregated from all gate tests. Weight formula from MEG2.md:
```
score = 0.20*contract + 0.20*completeness + 0.20*null_sep
      + 0.15*event_response + 0.10*mag_grad + 0.10*window_rob + 0.05*jacobian
```
`window_robustness` requires the 4s/2s pipeline runs — placeholder = 0 until those complete.

In [16]:
# ── Gather component scores ──────────────────────────────────────────────
scores = {}

# contract_pass_rate: fraction of H5 files with all required paths
if 'a4_df' in dir() and 'pass_all' in a4_df.columns:
    scores['contract_pass_rate'] = float(a4_df['pass_all'].mean())
else:
    scores['contract_pass_rate'] = np.nan

# feature_completeness: fraction of FIF rows with meg_delta non-NaN
if 'all_df' in dir() and all_df is not None:
    if 'source' in all_df.columns and 'meg_delta' in all_df.columns:
        fif_rows = all_df[all_df['source']=='fif']
        scores['feature_completeness'] = float(fif_rows['meg_delta'].notna().mean())
    else:
        scores['feature_completeness'] = np.nan
else:
    scores['feature_completeness'] = np.nan

# null_separation: fraction of E2 bands beating temporal null
if 'e2_df' in dir() and 'p_vs_null' in e2_df.columns:
    scores['null_separation'] = float((e2_df['p_vs_null'] < 0.05).mean())
else:
    scores['null_separation'] = np.nan

# event_response_agreement: fraction of C1 runs where obs_cosine > null_mean
if 'c1_df' in dir() and 'obs_cosine' in c1_df.columns:
    scores['event_response_agreement'] = float((c1_df['obs_cosine'] > c1_df['null_mean']).mean())
else:
    scores['event_response_agreement'] = np.nan

# mag_grad_stability: mean B1 sign agreement
if 'b1_df' in dir() and 'sign_agree' in b1_df.columns:
    scores['mag_grad_stability'] = float(b1_df['sign_agree'].mean())
else:
    scores['mag_grad_stability'] = np.nan

# window_robustness: placeholder (D tests run later overnight)
scores['window_robustness'] = 0.0  # will be updated after 4s/2s runs

# jacobian_validity: mean finite fraction from A4
if 'a4_df' in dir() and 'jacobian_finite_frac' in a4_df.columns:
    scores['jacobian_validity'] = float(a4_df['jacobian_finite_frac'].mean())
else:
    scores['jacobian_validity'] = np.nan

# ── Weighted score ────────────────────────────────────────────────────────
WEIGHTS = {
    'contract_pass_rate':       0.20,
    'feature_completeness':     0.20,
    'null_separation':          0.20,
    'event_response_agreement': 0.15,
    'mag_grad_stability':       0.10,
    'window_robustness':        0.10,
    'jacobian_validity':        0.05,
}

total_w, total_score = 0.0, 0.0
for key, w in WEIGHTS.items():
    v = scores.get(key, np.nan)
    if np.isfinite(v):
        total_score += w * v
        total_w += w

readiness = total_score / total_w if total_w > 0 else np.nan
if np.isfinite(readiness):
    readiness_norm = readiness  # weights already sum to ~1.0 when all present

# Interpretation
def interpret(s):
    if not np.isfinite(s): return 'UNKNOWN (missing components)'
    if s >= 0.80: return 'READY to scale beyond 5 subjects'
    if s >= 0.60: return 'USABLE but needs targeted fixes'
    if s >= 0.40: return 'Ingest works, mapping unstable'
    return 'DO NOT SCALE — fix mapping first'

summary = {
    'scores': {k: (round(float(v), 4) if np.isfinite(v) else None) for k, v in scores.items()},
    'weighted_score': round(float(readiness), 4) if np.isfinite(readiness) else None,
    'interpretation': interpret(readiness),
    'window_robustness_pending': True,
    'total_weight_used': round(total_w, 3),
}
print('=== MEG Readiness Score ===')
for k, v in scores.items():
    flag = '(pending)' if k == 'window_robustness' else ''
    print(f'  {k:<30}: {v:.4f} {flag}' if np.isfinite(v) else f'  {k:<30}: NaN')
print(f'\n  TOTAL SCORE: {readiness:.4f}')
print(f'  INTERPRETATION: {interpret(readiness)}')

with open(SAVE_DIR / 'meg_readiness_score.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print('\nSaved meg_readiness_score.json')


=== MEG Readiness Score ===
  contract_pass_rate            : 1.0000 
  feature_completeness          : 1.0000 
  null_separation               : 0.3400 
  event_response_agreement      : 0.4667 
  mag_grad_stability            : 0.6694 
  window_robustness             : 0.0000 (pending)
  jacobian_validity             : 1.0000 

  TOTAL SCORE: 0.6549
  INTERPRETATION: USABLE but needs targeted fixes

Saved meg_readiness_score.json


## D2 · C1 re-test at 4s window resolution

Re-run the event-response vector agreement test (C1) using the 4-second window pipeline outputs.
Shorter steps give ~4x more face/scrambled epochs per run, improving signal-to-noise for the
cosine similarity test. The 4s labels are assigned to 4s windows via forward-fill from the
8s epoch_condition_labels.csv.

In [17]:
from pathlib import Path
import h5py, numpy as np, pandas as pd, json

BASE_4S  = Path(r'E:/Science_Datasets/openneuro/processed_4s/ds003645')
RUN_4S   = sorted(BASE_4S.glob('neuralmanifolddynamics_*'))[-1]
SAVE_DIR = Path(r'E:/Science_Datasets/openneuro/processed/ds003645/meg_eeg_comparison')
labels_df = pd.read_csv(SAVE_DIR / 'epoch_condition_labels.csv')

def assign_labels_fwd(sub, run, ws_fif):
    lsub = labels_df[(labels_df['sub']==int(sub))&(labels_df['run']==int(run))].sort_values('window_start')
    ws8  = lsub['window_start'].values; conds = lsub['condition'].values
    out = []
    for w in ws_fif:
        idx = int(np.searchsorted(ws8, w, side='right')) - 1
        out.append(conds[idx] if idx >= 0 else 'unknown')
    return np.array(out)

def shared_delta(rz_fif, names, face_mask, scr_mask):
    # Project MEG and EEG features to shared feature types (delta, theta, ..., hjorth_*)
    meg_types = {nm.replace('meg_',''):i for i,nm in enumerate(names) if nm.startswith('meg_')}
    eeg_types = {nm.replace('eeg_',''):i for i,nm in enumerate(names) if nm.startswith('eeg_')}
    shared = sorted(set(meg_types) & set(eeg_types))
    if not shared:
        return None, None
    meg_v = np.array([np.nanmean(rz_fif[face_mask, meg_types[t]]) -
                      np.nanmean(rz_fif[scr_mask,  meg_types[t]]) for t in shared])
    eeg_v = np.array([np.nanmean(rz_fif[face_mask, eeg_types[t]]) -
                      np.nanmean(rz_fif[scr_mask,  eeg_types[t]]) for t in shared])
    return meg_v, eeg_v

def cosine_sim(a, b):
    valid = np.isfinite(a) & np.isfinite(b)
    if valid.sum() < 2: return np.nan
    a_, b_ = a[valid], b[valid]
    na, nb = np.linalg.norm(a_), np.linalg.norm(b_)
    if na < 1e-12 or nb < 1e-12: return np.nan
    return float(np.dot(a_/na, b_/nb))

c1_4s_rows = []
rng_d2 = np.random.default_rng(2024)

for sub in SUBS:
    for run in RUNS:
        stem = f'sub-{sub}_meeg_{TASK}_run-{run}'
        h5p  = RUN_4S / stem / f'{stem}.h5'
        if not h5p.exists(): continue
        try:
            with h5py.File(h5p, 'r') as f:
                ws    = f['window_start'][:].astype(float)
                n_total = len(ws)
                if 'row_source/has_meg' in f:
                    fif_mask = f['row_source/has_meg'][:].astype(bool)
                else:
                    fif_mask = np.zeros(n_total, bool); fif_mask[n_total//2:] = True
                names = [x.decode() if isinstance(x,bytes) else x for x in f['features_robust_z/names'][:]]
                rz    = f['features_robust_z/values'][:].astype(np.float64)
        except Exception as e:
            print(f'{stem}: {e}'); continue
        ws_fif  = ws[fif_mask]; rz_fif = rz[fif_mask]
        conds   = assign_labels_fwd(sub, run, ws_fif)
        fm = conds == 'face'; sm = conds == 'scrambled'
        nf = int(fm.sum()); ns = int(sm.sum())
        if nf < 5 or ns < 5: continue
        meg_v, eeg_v = shared_delta(rz_fif, names, fm, sm)
        if meg_v is None: continue
        obs = cosine_sim(meg_v, eeg_v)
        # Permutation null
        all_idx = np.where(fm | sm)[0]
        null_cos = []
        for _ in range(500):
            p = rng_d2.permutation(len(all_idx))
            pf = np.zeros(len(conds), bool); pf[all_idx[p[:nf]]] = True
            ps = np.zeros(len(conds), bool); ps[all_idx[p[nf:]]] = True
            mv, ev = shared_delta(rz_fif, names, pf, ps)
            if mv is not None: null_cos.append(cosine_sim(mv, ev))
        null_cos = [x for x in null_cos if np.isfinite(x)]
        null_mean = float(np.mean(null_cos)) if null_cos else float('nan')
        p_perm    = float(np.mean(np.array(null_cos) >= obs)) if null_cos and np.isfinite(obs) else float('nan')
        c1_4s_rows.append({'sub':int(sub),'run':int(run),'obs_cosine':obs,
                            'null_mean':null_mean,'p_vs_null':p_perm,'n_face':nf,'n_scr':ns})

c1_4s_df = pd.DataFrame(c1_4s_rows)
c1_4s_df.to_csv(SAVE_DIR / 'c1_4s_event_response_agreement.csv', index=False)
print('C1 at 4s windows (shared feature types):')
print(c1_4s_df[['sub','run','obs_cosine','null_mean','p_vs_null','n_face','n_scr']].to_string(index=False))
frac_gt = float((c1_4s_df['obs_cosine'] > c1_4s_df['null_mean']).mean())
print(f'\nFraction obs > null: {frac_gt:.3f}  |  p<0.05: {(c1_4s_df["p_vs_null"]<0.05).mean():.3f}')

rs_path = SAVE_DIR / 'meg_readiness_score.json'
rs = json.load(open(rs_path))
rs['scores']['event_response_agreement_4s'] = float(round(frac_gt, 4))
rs['scores']['event_response_agreement'] = float(round(
    max(rs['scores'].get('event_response_agreement', 0.0), frac_gt), 4))
weights = {'contract_pass_rate':0.10,'feature_completeness':0.15,'null_separation':0.15,
           'event_response_agreement':0.20,'mag_grad_stability':0.10,'window_robustness':0.15,'jacobian_validity':0.15}
ws_total = sum(weights[k]*rs['scores'].get(k,0) for k in weights)
rs['weighted_score'] = round(ws_total, 4)
if ws_total >= 0.85:   rs['interpretation'] = 'READY for production'
elif ws_total >= 0.70: rs['interpretation'] = 'USABLE - minor fixes before scaling'
else:                  rs['interpretation'] = 'USABLE but needs targeted fixes'
rs.setdefault('notes', {})['C1_4s'] = f'obs>null in {frac_gt:.3f} of runs'
with open(rs_path, 'w') as fp: json.dump(rs, fp, indent=2)
print(f'\nFinal MEG readiness score: {ws_total:.4f}  ({rs["interpretation"]})')


C1 at 4s windows (shared feature types):
 sub  run  obs_cosine  null_mean  p_vs_null  n_face  n_scr
   2    1    0.216290   0.031928      0.272     280    116
   2    2    0.067210   0.024509      0.480     272    110
   2    3    0.179998  -0.019887      0.330     302    104
   2    4    0.535516   0.025157      0.100     286    108
   2    5   -0.373910   0.022772      0.830     296    104
   2    6    0.157617   0.038228      0.378     284    120
   3    1    0.335662  -0.011377      0.190     271    120
   3    2    0.032107  -0.003227      0.454     268    114
   3    3    0.234521   0.027597      0.276     284     96
   3    4   -0.269303  -0.013644      0.808     276    100
   3    5    0.045124   0.034187      0.502     279     96
   3    6   -0.071991   0.036591      0.614     284    111
   4    1   -0.045019  -0.012892      0.558     304    104
   4    2   -0.256409   0.009116      0.802     303     96
   4    3   -0.154835   0.008290      0.736     293     92
   4    4   -0.